# 06 · AI Smog Explainer: bilingual advisories from the model's numbers

<a href="https://colab.research.google.com/github/FarhanRajputFelix/saans/blob/main/notebooks/06_llm_advisories.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The second LLM use in SAANS is in the product. For a city and day, the LLM receives the forecast, the AQI, the hazard probability and the top drivers from SHAP, and writes a three-line advisory in **English and Urdu**.

Guardrails:
1. **The LLM never predicts.** It only rewords numbers produced by the models.
2. **Automatic number check.** Every number in the English text must be one we supplied (or a fixed term such as PM2.5 or N95). Advisories that fail are discarded, and the dashboard falls back to a rule-based template.
3. **Cached once per city per day** (`app/data/advisories.json`), so the dashboard makes no LLM calls and needs no key.

**Setup.** Runs the same way locally and in Google Colab. In Colab it clones the repo (which includes the cleaned data) and reads API keys from Colab's *Secrets* panel (key icon on the left).

In [1]:
import os, sys, subprocess
if "google.colab" in sys.modules:
    if not os.path.exists("/content/saans"):
        subprocess.run(["git", "clone", "-q", "https://github.com/FarhanRajputFelix/saans.git", "/content/saans"], check=True)
    os.chdir("/content/saans")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm", "shap", "python-dotenv"], check=True)
    from google.colab import userdata
    for k in ["OPENAQ_API_KEY", "FIRMS_MAP_KEY", "GEMINI_API_KEY", "GROQ_API_KEY"]:
        try:
            os.environ[k] = userdata.get(k)
        except Exception:
            pass
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # work from the repo root so `saans` and `data/` resolve
sys.path.insert(0, os.getcwd())

In [2]:
import json
import re
import pandas as pd

from saans.config import ROOT, CITIES
from saans.aqi import BREAKPOINTS, pm25_to_aqi
from saans import llm

pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 200)
DASH = ROOT / "app" / "data"
daily = pd.read_parquet(DASH / "forecasts_daily.parquet")
daily["date"] = daily.time_pkt.dt.tz_convert("Asia/Karachi").dt.date.astype(str)
print(f"{len(daily)} daily forecasts available")

930 daily forecasts available


## Which days get an LLM advisory

Free LLM quotas are limited, so advisories are generated for the days that matter most: every city on the dashboard's two default dates, plus each city's three highest-forecast days per season. All other days use the template.

In [3]:
mid_2025 = sorted(daily[daily.season == "2025-26"].date.unique())
picks = set(daily.index[daily.date.isin(["2024-11-06", mid_2025[len(mid_2025) // 2]])])
for _, g in daily.groupby(["season", "city"]):
    picks |= set(g.nlargest(3, "forecast_24h").index)
todo = daily.loc[sorted(picks)]
print(f"{len(todo)} advisories to write")

32 advisories to write


In [4]:
PROMPT = """You are the "AI Smog Explainer" of SAANS, a smog forecasting system for Pakistan.
Write a health advisory for residents of {city} for the next 24 hours (forecast issued {date}, 08:00 Pakistan time).

Use ONLY these numbers, exactly as written. Do not calculate, round differently or invent any other number:
- forecast 24-hour mean PM2.5: {f} micrograms per cubic metre
- US EPA AQI: {aqi} ({category})
- chance that the next 24 hours are hazardous (AQI above 300): {p}%
- main reasons, from the model's explanation: {drivers}

Write exactly three short lines in plain English:
1. the forecast and the AQI;
2. why, naming the main reasons in everyday words;
3. what people should do, with care for children, the elderly and people with asthma.
Then write the same three lines in natural, simple Urdu (Urdu script).

Return JSON: {{"en": "<three lines separated by \\n>", "ur": "<three lines separated by \\n>"}}"""

def category(f):
    return next((c for lo, hi, a, b, c in BREAKPOINTS if f <= hi + 0.05), "Hazardous")

def facts(r):
    f = int(round(r.forecast_24h))
    drivers = [d["label"].lower() for d in json.loads(r.top_drivers)]
    return {"city": CITIES[r.city].name, "date": r.date, "f": f, "aqi": int(pm25_to_aqi(r.forecast_24h)),
            "category": category(r.forecast_24h), "p": int(round(100 * r.hazard_prob)), "drivers": "; ".join(drivers)}

URDU_DIGITS = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")

def numbers_ok(text: str, fx: dict) -> bool:
    """Every number in the advisory must be one we supplied (Urdu digits are normalised first)."""
    allowed = {str(fx["f"]), str(fx["aqi"]), str(fx["p"]), "2.5", "24", "300", "95", "8", "08", "00", "1", "2", "3"}
    allowed |= set(re.findall(r"\d+(?:\.\d+)?", fx["date"]))
    return all(n in allowed for n in re.findall(r"\d+(?:\.\d+)?", text.translate(URDU_DIGITS)))

print(PROMPT.format(**facts(todo.iloc[0])))

You are the "AI Smog Explainer" of SAANS, a smog forecasting system for Pakistan.
Write a health advisory for residents of Islamabad for the next 24 hours (forecast issued 2024-11-06, 08:00 Pakistan time).

Use ONLY these numbers, exactly as written. Do not calculate, round differently or invent any other number:
- forecast 24-hour mean PM2.5: 87 micrograms per cubic metre
- US EPA AQI: 173 (Unhealthy)
- chance that the next 24 hours are hazardous (AQI above 300): 0%
- main reasons, from the model's explanation: average pm2.5 over the last 24 hours; pm2.5 right now; time of year

Write exactly three short lines in plain English:
1. the forecast and the AQI;
2. why, naming the main reasons in everyday words;
3. what people should do, with care for children, the elderly and people with asthma.
Then write the same three lines in natural, simple Urdu (Urdu script).

Return JSON: {"en": "<three lines separated by \n>", "ur": "<three lines separated by \n>"}


In [5]:
advisories, rows = {}, []
for i, r in todo.iterrows():
    fx = facts(r)
    try:
        answer, provider = llm.ask(PROMPT.format(**fx), temperature=0.4, as_json=True)
        out = llm.parse_json(answer)
        ok = numbers_ok(out["en"], fx) and numbers_ok(out["ur"], fx)
    except Exception as e:
        out, provider, ok = {"en": "", "ur": ""}, f"failed: {e}"[:60], False
    rows.append({"city": r.city, "date": r.date, "forecast": fx["f"], "provider": provider, "numbers_check": ok})
    if ok:
        advisories[f"{r.city}|{r.date}"] = {"en": out["en"], "ur": out["ur"], "source": provider}
log = pd.DataFrame(rows)
(DASH / "advisories.json").write_text(json.dumps(advisories, ensure_ascii=False, indent=1), encoding="utf-8")
print(f"{len(advisories)} of {len(log)} advisories passed the number check and were cached; providers: {log.provider.value_counts().to_dict()}")
log[~log.numbers_check]

32 of 32 advisories passed the number check and were cached; providers: {'gemini:gemini-3.5-flash-lite': 32}


,city,date,forecast,provider,numbers_check


## Example: Lahore, 6 November 2024

In [6]:
ex = advisories.get("lahore|2024-11-06")
if ex:
    print(ex["en"]); print(); print(ex["ur"])

The forecast 24-hour mean PM2.5 is 386 micrograms per cubic metre, giving a US EPA AQI of 621 (Hazardous), with a 92% chance that the next 24 hours are hazardous (AQI above 300).
This is mainly driven by the average pm2.5 over the last 24 hours, pm2.5 right now, and upwind crop fires from the last 24 hours.
Everyone should stay indoors, especially children, the elderly, and people with asthma who need to take extra care.

اگلے 24 گھنٹوں کے لیے PM2.5 کی اوسط پیشگوئی 386 مائیکرو گرام فی مکعب میٹر ہے، جس سے US EPA AQI 621 (خطرناک) ہے، اور اگلے 24 گھنٹوں کے خطرناک (AQI 300 سے اوپر) ہونے کا امکان 92% ہے۔
اس کی بنیادی وجوہات پچھلے 24 گھنٹوں کی اوسط PM2.5، اس وقت کی PM2.5، اور پچھلے 24 گھنٹوں کی اوپر والی ہوا کے فصلوں کو آگ لگانے کے واقعات ہیں۔
تمام لوگ گھروں کے اندر رہیں، خاص طور پر بچے، بزرگ، اور وہ افراد جنہیں دممہ (asthma) ہے، وہ بالخصوص احتیاط کریں۔
